In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,patient id,patient age,patient gender,hepatectomy,tissue,paired nontumor,tumor stage
samples,,,,,,,,
GSM4462456,Sample_1 [Cat_N001],PTN001,17,M,NA,"Normal tissue, liver",NA,NaN
GSM4462457,Sample_2 [Cat_N002],PT076,54,M,NA,"Normal tissue, liver",NA,NaN
GSM4462458,Sample_3 [Cat_N003],PT075_N,64,M,NA,"Normal tissue, liver",NA,NaN
GSM4462459,Sample_4 [Cat_N004],PT074,50,F,NA,"Normal tissue, liver",NA,NaN
GSM4462460,Sample_5 [Cat_N005],PT077,73,M,NA,"Normal tissue, liver",NA,NaN
...,...,...,...,...,...,...,...,...
GSM4462579,Sample_73 [Cat_HPC-003],PT072_rep1,48,M,partial_hepatectomy,"Gastric cancer, liver",NA,combined HCC and intrahepatic cholangiocarcino...
GSM4462580,Sample_88 [Cat_HPC-003r],PT072_rep2,48,M,partial_hepatectomy,"Gastric cancer, liver",NA,combined HCC and intrahepatic cholangiocarcino...
GSM4462581,Sample_74 [Cat_HPC-004],PT021,61,M,NA,"Gastric cancer, liver",NA,combined HCC and intrahepatic cholangiocarcino...


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
df_combined['sample_title'] = df_combined['sample_title'].str.extract(r'\[([^\]]+)\]').squeeze().str.replace('^Cat_', '', regex=True)
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_95413/1508821078.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,patient id,patient age,patient gender,hepatectomy,tissue,paired nontumor,tumor stage,DDX11L1,WASH7P,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM4462456,N001,PTN001,17,M,NA,"Normal tissue, liver",NA,NaN,2,66,...,13801,47,15,17,6787,620,212,9770,120,385
GSM4462457,N002,PT076,54,M,NA,"Normal tissue, liver",NA,NaN,3,65,...,26902,58,25,31,16683,1219,510,25787,163,564
GSM4462458,N003,PT075_N,64,M,NA,"Normal tissue, liver",NA,NaN,3,61,...,14933,66,16,31,8032,949,236,13750,258,597
GSM4462459,N004,PT074,50,F,NA,"Normal tissue, liver",NA,NaN,3,73,...,7107,18,5,6,3602,327,105,5570,75,209
GSM4462460,N005,PT077,73,M,NA,"Normal tissue, liver",NA,NaN,3,117,...,14265,50,17,24,7645,998,297,11613,210,755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM4462579,HPC-003,PT072_rep1,48,M,partial_hepatectomy,"Gastric cancer, liver",NA,combined HCC and intrahepatic cholangiocarcino...,3,87,...,17617,67,26,33,14833,1007,327,14332,88,255
GSM4462580,HPC-003r,PT072_rep2,48,M,partial_hepatectomy,"Gastric cancer, liver",NA,combined HCC and intrahepatic cholangiocarcino...,1,88,...,20969,82,24,30,18229,1372,425,18293,126,481
GSM4462581,HPC-004,PT021,61,M,NA,"Gastric cancer, liver",NA,combined HCC and intrahepatic cholangiocarcino...,6,136,...,5539,31,11,14,6062,535,190,6303,67,230


In [3]:
import pymupdf

from openai import OpenAI
from typing import Optional, List, Dict, Any

client = OpenAI(api_key='os.environ.get("OPENAI_API_KEY")')


def call_openai(
    user_text: str,
    *,
    system_instructions: Optional[str] = None,
    model: str = "gpt-5-mini-2025-08-07",
    verbosity: str = "low",
    reasoning_effort: str = "low",
) -> str:
    
    messages: List[Dict[str, Any]] = []
    if system_instructions:
        messages.append({
            "role": "system",
            "content": [{"type": "input_text", "text": system_instructions}],
        })
    messages.append({
        "role": "user",
        "content": [{"type": "input_text", "text": user_text}],
    })

    resp = client.responses.parse(
        model=model,
        input=messages,
        text={"verbosity": verbosity},
        reasoning={"effort": reasoning_effort},
        service_tier="flex",
    )
    return resp


prompt = """Extract Supplementary Table 2 from the following PDF text. The table may be split across multiple pages/parts. 
Combine all parts into a single unified CSV format.

Instructions:
- Identify all parts of Supplementary Table 2 in the text
- Extract the table headers and all data rows
- Merge the parts into one complete table
- Output as valid CSV format with proper column headers
- Preserve all data values exactly as they appear

PDF Text:
{text}

Output the complete Supplementary Table 2 as CSV:"""


# doc = pymupdf.open("41698_2021_167_MOESM1_ESM.pdf")
# # Extract text from the 17th page
# pdf_text = doc[16].get_text() # "\n\n".join([page.get_text() for page in doc[17:18]])
# doc.close()

# result = call_openai(
#     prompt.format(text=pdf_text),
#     system_instructions="Follow the instructions carefully. Do only what is asked.",
#     reasoning_effort="medium"
# )

# # Extract text from response
# if result.output and len(result.output) > 1:
#     message = result.output[1]
#     if hasattr(message, 'content') and message.content:   
#         # Save to CSV
#         with open('table_s2_clinical_data.csv', 'w') as f:
#             f.write(message.content[0].text)

In [4]:
df_clinical_data = pd.read_csv('table_s2_clinical_data.csv')

# Merge the dataframes
df_combined = df_clinical_data.merge(
    df_combined,
    left_on='Sample',
    right_on='sample_title',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)
df_combined

,Sample,Op,Pathology,Risk factor,Progression,Tumor stage,TNM N,TNM M,Age,Sex,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,DH013,TH,DN high,HBV,NaN,Stage II,N0,M0,55,M,...,12022,26,11,13,10712,1066,454,14578,61,649
1,TG1-003,TH,ES1,HBV,NaN,Stage II,N0,M0,66,M,...,14244,18,5,7,9641,727,263,14269,105,478
2,TG1-005,TH,ES1,HBV,NaN,Stage II,N0,M0,41,M,...,13319,72,23,41,12035,1112,323,15862,314,521
3,TG1-007,TH,ES1,HCV,NaN,Stage II,N0,M0,60,M,...,13934,22,6,10,12769,1070,471,19755,116,474
4,TG1-009,TH,ES1,HBV,NaN,Stage I,N0,M0,53,M,...,17853,16,4,16,9065,1120,376,12468,103,443
5,TG1-012,TH,ES1,HBV,NaN,Stage II,N0,M0,43,F,...,23229,20,4,8,13267,1004,365,21064,93,459
6,TG1-013,TH,ES1,HBV,NaN,Stage I,N0,M0,52,F,...,14662,10,3,4,7915,623,277,13985,58,288
7,TG1-017,TH,ES1,HBV,NaN,Stage II,Nx,Mx,58,M,...,7103,13,7,11,5244,451,206,7823,67,236
8,TG1-021,TH,ES1,HBV,NaN,Stage II,N0,M0,60,M,...,13231,12,2,6,8028,567,249,9424,98,345
9,TG1-023,TH,ES1,HBV,NaN,Stage IIIA,N0,M0,53,M,...,23345,210,32,61,22642,2195,643,21630,274,999


# Generate description


In [5]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'table_s2_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
